# LightOnOCR-2 Magyar Fine-tuning (v6 - Full Augmentation)

**Runtime → Change runtime type → T4 GPU**

- 20+ font (Arial, Times, OCR-B, Courier, stb.)
- Augmentációk: zaj, forgatás, torzítás, elmosás

In [ ]:
# 1. Telepítés
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow opencv-python-headless

# Fontok letöltése
!mkdir -p /content/fonts

# Sans-serif (Arial-szerű)
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSans-Regular.ttf -O /content/fonts/LiberationSans-Regular.ttf
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSans-Bold.ttf -O /content/fonts/LiberationSans-Bold.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf -O /content/fonts/NotoSans-Regular.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Bold.ttf -O /content/fonts/NotoSans-Bold.ttf
!wget -q https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSans.ttf -O /content/fonts/DejaVuSans.ttf
!wget -q https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Regular.ttf -O /content/fonts/Roboto-Regular.ttf
!wget -q https://github.com/googlefonts/opensans/raw/main/fonts/ttf/OpenSans-Regular.ttf -O /content/fonts/OpenSans-Regular.ttf
!wget -q https://github.com/adobe-fonts/source-sans/raw/release/TTF/SourceSans3-Regular.ttf -O /content/fonts/SourceSans3-Regular.ttf
!wget -q https://github.com/mozilla/Fira/raw/master/ttf/FiraSans-Regular.ttf -O /content/fonts/FiraSans-Regular.ttf
!wget -q https://github.com/IBM/plex/raw/master/IBM-Plex-Sans/fonts/complete/ttf/IBMPlexSans-Regular.ttf -O /content/fonts/IBMPlexSans-Regular.ttf

# Serif (Times-szerű)
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSerif-Regular.ttf -O /content/fonts/LiberationSerif-Regular.ttf
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSerif-Bold.ttf -O /content/fonts/LiberationSerif-Bold.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSerif/NotoSerif-Regular.ttf -O /content/fonts/NotoSerif-Regular.ttf
!wget -q https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSerif.ttf -O /content/fonts/DejaVuSerif.ttf
!wget -q https://github.com/adobe-fonts/source-serif/raw/release/TTF/SourceSerif4-Regular.ttf -O /content/fonts/SourceSerif4-Regular.ttf

# Monospace (Courier-szerű)
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationMono-Regular.ttf -O /content/fonts/LiberationMono-Regular.ttf
!wget -q https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSansMono.ttf -O /content/fonts/DejaVuSansMono.ttf
!wget -q https://github.com/IBM/plex/raw/master/IBM-Plex-Mono/fonts/complete/ttf/IBMPlexMono-Regular.ttf -O /content/fonts/IBMPlexMono-Regular.ttf

# OCR fontok
!wget -q https://github.com/nicokempe/ocrb-webfont/raw/master/OCRB-Regular.ttf -O /content/fonts/OCRB-Regular.ttf 2>/dev/null || true

print("✓ Telepítés kész")
!ls /content/fonts/*.ttf | wc -l

In [ ]:
# 2. Augmentációs függvények
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import random

def add_noise(img, intensity=0.02):
    """Gaussian zaj hozzáadása (scan artifact)"""
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, intensity * 255, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def add_salt_pepper(img, amount=0.005):
    """Salt & pepper zaj (bitmap corruption)"""
    arr = np.array(img)
    # Salt
    salt = np.random.random(arr.shape[:2]) < amount/2
    arr[salt] = 255
    # Pepper
    pepper = np.random.random(arr.shape[:2]) < amount/2
    arr[pepper] = 0
    return Image.fromarray(arr)

def rotate_image(img, max_angle=2.0):
    """Kis szögű forgatás (ferde scan)"""
    angle = random.uniform(-max_angle, max_angle)
    return img.rotate(angle, fillcolor='white', expand=False)

def perspective_transform(img, intensity=0.02):
    """Perspektíva torzítás (grid distortion)"""
    arr = np.array(img)
    h, w = arr.shape[:2]
    
    # Forrás pontok (sarkok)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    
    # Cél pontok (kis random eltolással)
    offset = int(min(w, h) * intensity)
    dst = np.float32([
        [random.randint(0, offset), random.randint(0, offset)],
        [w - random.randint(0, offset), random.randint(0, offset)],
        [w - random.randint(0, offset), h - random.randint(0, offset)],
        [random.randint(0, offset), h - random.randint(0, offset)]
    ])
    
    M = cv2.getPerspectiveTransform(src, dst)
    result = cv2.warpPerspective(arr, M, (w, h), borderValue=(255, 255, 255))
    return Image.fromarray(result)

def add_blur(img, radius=0.5):
    """Enyhe elmosás (rossz minőségű scan)"""
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def adjust_brightness(img, factor_range=(0.9, 1.1)):
    """Fényerő változtatás"""
    factor = random.uniform(*factor_range)
    arr = np.array(img).astype(np.float32)
    arr = np.clip(arr * factor, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def adjust_contrast(img, factor_range=(0.9, 1.1)):
    """Kontraszt változtatás"""
    factor = random.uniform(*factor_range)
    arr = np.array(img).astype(np.float32)
    mean = arr.mean()
    arr = np.clip((arr - mean) * factor + mean, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def apply_augmentations(img, aug_probability=0.7):
    """Véletlenszerű augmentációk alkalmazása"""
    if random.random() > aug_probability:
        return img  # Néha eredeti képet hagyunk
    
    augmentations = [
        (0.3, lambda x: add_noise(x, random.uniform(0.01, 0.03))),
        (0.2, lambda x: add_salt_pepper(x, random.uniform(0.002, 0.008))),
        (0.4, lambda x: rotate_image(x, random.uniform(0.5, 2.5))),
        (0.2, lambda x: perspective_transform(x, random.uniform(0.01, 0.03))),
        (0.2, lambda x: add_blur(x, random.uniform(0.3, 0.8))),
        (0.3, lambda x: adjust_brightness(x)),
        (0.3, lambda x: adjust_contrast(x)),
    ]
    
    for prob, aug_func in augmentations:
        if random.random() < prob:
            img = aug_func(img)
    
    return img

# Teszt
print("Augmentáció teszt:")
test_img = Image.new("RGB", (400, 100), "white")
draw = ImageDraw.Draw(test_img)
draw.text((20, 30), "őűŐŰ Teszt 123", fill="black")

fig, axes = [], []
from IPython.display import display
print("Eredeti:")
display(test_img)
print("\nAugmentált verziók:")
for i in range(4):
    aug_img = apply_augmentations(test_img.copy(), aug_probability=1.0)
    display(aug_img)

In [ ]:
# 3. Fontok betöltése
from pathlib import Path

FONT_DIR = Path("/content/fonts")
TEST_TEXT = "őűŐŰ öüóőúéáűí"

FONTS = []
for font_path in sorted(FONT_DIR.glob("*.ttf")):
    try:
        font = ImageFont.truetype(str(font_path), 24)
        test_img = Image.new("RGB", (100, 30), "white")
        ImageDraw.Draw(test_img).text((5, 5), "őű", fill="black", font=font)
        if any(p != (255,255,255) for p in test_img.getdata()):
            FONTS.append((font_path.stem, str(font_path)))
            print(f"✓ {font_path.stem}")
    except:
        pass

print(f"\n✓ {len(FONTS)} font betöltve")

In [ ]:
# 4. Adatgenerálás augmentációkkal
import json

HUNGARIAN_WORDS = [
    # ő
    "őr", "őriz", "ők", "ősz", "ősi", "őszinte", "őrült",
    "erő", "idő", "mező", "tető", "fő", "nő", "bő", "hő",
    "belső", "külső", "felső", "alsó", "utolsó", "első",
    "költő", "festő", "vezető", "börtön", "könyv", "között",
    "Győr", "dőlt", "dől", "töröl", "pörög", "görög", "örök",
    # ű
    "űr", "űrlap", "gyűrű", "tűz", "fűz", "gyűjt", "gyűlés",
    "tűnik", "fűszer", "hűtő", "hűvös", "hűség",
    "szürke", "szűk", "szűr", "sűrű", "bűvös", "működik", "műszer",
    # Dokumentum
    "Csatornadíj", "vízdíj", "díj", "tükörfúrógép", "árvíztűrő",
    "halványszürke", "fizetendő", "összeg", "összesen",
    "adószám", "cégjegyzékszám", "azonosító", "határidő",
]

TEMPLATES = [
    "Fizetendő összeg: {amt} Ft",
    "Csatornadíj: {amt} Ft",
    "Vízdíj: {amt} Ft",
    "Kedvezmény: {amt} Ft",
    "Adószám: {tax}",
    "Határidő: {date}",
    "Azonosító: {id}",
    "IBAN: HU{iban}",
]

def gen_text():
    lines = []
    lines.append(" ".join(random.sample(HUNGARIAN_WORDS, random.randint(5, 8))))
    for _ in range(random.randint(3, 5)):
        t = random.choice(TEMPLATES).format(
            amt=f"{random.randint(1,99)} {random.randint(100,999):03d}",
            tax=f"{random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}",
            date=f"2025.{random.randint(1,12):02d}.{random.randint(1,28):02d}",
            id=f"ID-{random.randint(100000,999999)}",
            iban=f"{random.randint(10,99)} {random.randint(1000,9999)} {random.randint(1000,9999)} {random.randint(1000,9999)}",
        )
        lines.append(t)
    lines.append("öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ")
    lines.append("Árvíztűrő tükörfúrógép")
    return "\n".join(lines)

def render(text, font_path, font_size=24):
    font = ImageFont.truetype(font_path, font_size)
    lines = text.split("\n")
    h = len(lines) * int(font_size * 1.5) + 80
    bg = random.choice(["white", "#fafafa", "#f5f5f5", "#fffef0"])
    img = Image.new("RGB", (850, h), bg)
    draw = ImageDraw.Draw(img)
    y = 40
    for line in lines:
        draw.text((40, y), line, fill="black", font=font)
        y += int(font_size * 1.5)
    return img

# Generálás
Path("training_data/images").mkdir(parents=True, exist_ok=True)
annotations = []

NUM_SAMPLES = 800  # Több minta az augmentációk miatt
AUG_RATIO = 0.6    # 60% augmentált

print(f"Generálás: {NUM_SAMPLES} kép ({int(NUM_SAMPLES*AUG_RATIO)} augmentált)...")
for i in range(NUM_SAMPLES):
    text = gen_text()
    font_name, font_path = random.choice(FONTS)
    font_size = random.choice([18, 20, 22, 24, 26, 28])
    
    img = render(text, font_path, font_size)
    
    # Augmentáció alkalmazása
    if random.random() < AUG_RATIO:
        img = apply_augmentations(img, aug_probability=1.0)
        augmented = True
    else:
        augmented = False
    
    img.save(f"training_data/images/{i:05d}.png")
    annotations.append({"image": f"{i:05d}.png", "text": text, "font": font_name, "augmented": augmented})
    
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{NUM_SAMPLES}")

with open("training_data/annotations.jsonl", "w", encoding="utf-8") as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

aug_count = sum(1 for a in annotations if a["augmented"])
print(f"\n✓ {NUM_SAMPLES} kép ({aug_count} augmentált, {NUM_SAMPLES-aug_count} eredeti)")

In [ ]:
# 5. Példák
print("Példa képek:")
# Eredeti
for i, a in enumerate(annotations[:50]):
    if not a["augmented"]:
        print(f"\nEredeti ({a['font']}):")
        display(Image.open(f"training_data/images/{a['image']}"))
        break

# Augmentált
for i, a in enumerate(annotations[:50]):
    if a["augmented"]:
        print(f"\nAugmentált ({a['font']}):")
        display(Image.open(f"training_data/images/{a['image']}"))
        break

In [ ]:
# 6. Modell
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

MODEL_ID = "lightonai/LightOnOCR-2-1B-base"
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.05)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 7. Dataset
from datasets import Dataset

def load_data():
    data = []
    with open("training_data/annotations.jsonl", encoding="utf-8") as f:
        for line in f:
            e = json.loads(line)
            data.append({"image_path": f"training_data/images/{e['image']}", "text": e["text"]})
    return Dataset.from_list(data)

def process(ex):
    img = Image.open(ex["image_path"]).convert("RGB")
    img_in = processor.image_processor(img, return_tensors="pt")
    txt_in = processor.tokenizer(ex["text"], return_tensors="pt", padding="max_length", max_length=512, truncation=True)
    return {
        "pixel_values": img_in["pixel_values"].squeeze(0),
        "input_ids": txt_in["input_ids"].squeeze(0),
        "attention_mask": txt_in["attention_mask"].squeeze(0),
        "labels": txt_in["input_ids"].squeeze(0),
    }

dataset = load_data().map(process, remove_columns=["image_path", "text"])
print(f"✓ Dataset: {len(dataset)}")

In [ ]:
# 8. Training
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./lighton-hun-lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=25,
    save_steps=150,
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
print(f"Tanítás: {len(dataset)} kép, 5 epoch (~25-35 perc)")
trainer.train()
print("✓ Kész!")

In [ ]:
# 9. Mentés
model.save_pretrained("./lighton-hun-lora")
merged = model.merge_and_unload()
merged.save_pretrained("./lighton-hun-merged")
processor.save_pretrained("./lighton-hun-merged")
print("✓ Mentve")

In [ ]:
# 10. Teszt
print("Teszt:")
for idx in [0, 200, 400, 600]:
    img = Image.open(f"training_data/images/{idx:05d}.png")
    inputs = processor.image_processor(img, return_tensors="pt")
    inputs = {k: v.to(merged.device) for k, v in inputs.items()}
    inputs["input_ids"] = processor.tokenizer("", return_tensors="pt")["input_ids"].to(merged.device)
    
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=400, do_sample=False)
    result = processor.tokenizer.decode(out[0], skip_special_tokens=True)
    
    print(f"\n=== #{idx} ===")
    display(img)
    print(result[:300])

In [ ]:
# 11. Letöltés
!zip -r lighton-hun-merged.zip lighton-hun-merged/
from google.colab import files
files.download("lighton-hun-merged.zip")

print("\n" + "="*50)
print("MAC-EN:")
print("="*50)
print("unzip lighton-hun-merged.zip")
print("mlx_vlm convert --hf-path lighton-hun-merged \\")
print("    --mlx-path models/lighton-hun-mlx -q --q-bits 4")